# VALORA AI: Better Data, Better Predictions
## Notebook 02: feature engineering and leakage protection

This notebook:
1. defines the three pre-specified feature sets **A (minimal)**, **B (location)** and **C (full property)**;
2. documents every feature decision (`results/feature_decisions.csv`);
3. measures redundancy: how much the "rich" features repeat information that is already in the minimal set;
4. proves the leakage rules (forbidden columns, transformations fitted on training rows only);
5. builds and checks the grouped cross-validation folds (`results/cv_folds.csv`).

**No feature is chosen by looking at model results.** The sets were fixed in `docs/EVALUATION_PROTOCOL.md` before any model was fitted.

**What this code does:** Sets up the paths and loads the modelling table from notebook 01.

**Why we need it:** Every notebook starts from the same saved table.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'budva_data.py').exists():      # works when started from notebooks/ or from the project root
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Start Jupyter inside the VALORA_BUDVA_APPRAISAL_EXPERIMENT folder')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 90)

import budva_data as bd
import budva_features as bf
import budva_eval as be
from budva_models import BASELINES, PricePerM2Baseline
m = bd.load_model_table()
print(m.shape)

(80, 31)


## 1. The three feature sets

| Set | Idea | Columns |
|---|---|---|
| **A: minimal** | what a quick €/m² valuation uses | area, neighbourhood, appraisal year |
| **B: location** | A + finer location | + distance to sea, distance to centre/main road, view type |
| **C: full property** | everything justified | + bedrooms, bathrooms, floor, building age, elevator, terrace, furnished, renovated, construction type, building permit, registered encumbrance |

**Hypothesis to test:** *Can a small, carefully selected set of meaningful features outperform a larger feature set?*

- **Why the bigger sets may not help:** with about 64 training properties per fold, every extra column is another parameter to estimate.

**What this code does:** Shows every column and how it is used, and saves the table.

**Why we need it:** One table documents each decision: which set uses a column, its transformation and the reason (or why it is excluded).

In [2]:
decisions = bf.feature_decisions()
decisions.to_csv(bd.RESULTS_DIR / 'feature_decisions.csv', index=False, encoding='utf-8', lineterminator='\n')
decisions

,column,used in sets,transformation,reason
0,area_m2,ABC,log (linear) / raw (trees),size is the main driver of value
1,neighborhood_group,ABC,"one-hot, categories seen < 2 times in training grouped as ""infrequent""",location
2,appraisal_year,ABC,standardised (linear) / raw (trees),values rose strongly 2021-2026
3,distance_to_sea_m,BC,log(1 + metres) (linear) / raw (trees),coastal premium
4,distance_to_center_road_m,BC,log(1 + metres) (linear) / raw (trees),access to the town centre
5,view_type,BC,"one-hot, categories seen < 2 times in training grouped as ""infrequent""",sea view premium
6,bedrooms,C,standardised (linear) / raw (trees),layout
7,bathrooms,C,standardised (linear) / raw (trees),layout / standard
8,floor,C,standardised (linear) / raw (trees),position in the building
9,building_age_years,C,standardised (linear) / raw (trees),age and standard of the building


## 2. Redundancy: what do the richer features add?

The richer sets help only if their columns carry information that is not already in area, neighbourhood and year. Three checks follow:
1. **Distances:** the share of their variation explained by the neighbourhood alone (R² of the neighbourhood means).
2. **Layout:** the correlation of bedrooms and bathrooms with area.
3. **Rare categories:** the minority counts of the binary columns.

**What this code does:** Computes the share of distance variation explained by neighbourhood, and the correlations of layout variables with area.

**Why we need it:** If a new feature is mostly a copy of an existing one, it can add noise but little signal.

In [3]:
def explained_by_neighbourhood(col, transform=lambda s: np.log1p(s.astype(float))):
    v = transform(m[col]).astype(float)
    fitted = v.groupby(m['neighborhood_group']).transform('mean')
    return 1 - ((v - fitted) ** 2).sum() / ((v - v.mean()) ** 2).sum()

for c in ['distance_to_sea_m', 'distance_to_center_road_m']:
    print(f'{c}: {100 * explained_by_neighbourhood(c):.1f}% of the variation of log(1 + distance) is explained by the neighbourhood')
print(f"sea view: {100 * explained_by_neighbourhood('view_type', lambda s: (s == 'sea').astype(float)):.1f}% explained by the neighbourhood")
print()
print(m[['area_m2', 'bedrooms', 'bathrooms', 'floor', 'building_age_years', 'appraisal_year']].corr().round(2).to_string())
print()
for c in ['elevator', 'terrace', 'furnished', 'renovated', 'building_permit', 'registered_encumbrance']:
    print(f'{c:24s} minority cases: {int(min((m[c] == 1).sum(), (m[c] == 0).sum()))}')

distance_to_sea_m: 93.8% of the variation of log(1 + distance) is explained by the neighbourhood
distance_to_center_road_m: 91.3% of the variation of log(1 + distance) is explained by the neighbourhood
sea view: 81.8% explained by the neighbourhood

                    area_m2  bedrooms  bathrooms  floor  building_age_years  appraisal_year
area_m2                1.00      0.76       0.79   0.01                0.04            0.24
bedrooms               0.76      1.00       0.58  -0.10                0.21            0.33
bathrooms              0.79      0.58       1.00   0.06                0.01            0.10
floor                  0.01     -0.10       0.06   1.00               -0.15            0.09
building_age_years     0.04      0.21       0.01  -0.15                1.00            0.08
appraisal_year         0.24      0.33       0.10   0.09                0.08            1.00

elevator                 minority cases: 21
terrace                  minority cases: 3
furnished         

**Reading the output:**
- **Distances and sea view:** most of their variation is already explained by the neighbourhood, so set B adds little new information.
- **Bedrooms and bathrooms:** strongly correlated with area.
- **Binary columns:** several have only 2-4 minority cases.
- **What to expect:** sets B and C mainly add **parameters**, not much new **information**. Whether that helps is tested in notebook 03.

## 3. Leakage rules

**What this code does:** Checks that no forbidden column is in any feature set, and shows why `price_per_m2_eur` is leakage.

**Why we need it:** If `price_per_m2_eur` were a feature, the model could rebuild the target exactly (value = €/m² × area).

In [4]:
for name, cols in bf.FEATURE_SETS.items():
    bad = set(cols) & set(bf.FORBIDDEN)
    assert not bad, (name, bad)
    print(f'set {name}: {len(cols)} columns, forbidden columns: none')
rebuilt = m['price_per_m2_eur'] * m['area_m2']
print('max |price_per_m2 × area − target|: €{:,.2f}'.format((rebuilt - m['market_value_eur']).abs().max()))

set A: 3 columns, forbidden columns: none
set B: 6 columns, forbidden columns: none
set C: 17 columns, forbidden columns: none
max |price_per_m2 × area − target|: €0.53


**What this code does:** Fits the preprocessing of each set on one training fold and lists the resulting model inputs.

**Why we need it:** It shows that encoders and scalers learn only from the training rows, and how unseen neighbourhoods are handled.

In [5]:
y = m['market_value_eur'].to_numpy(float)
groups = m['valuation_cluster'].to_numpy()
r, k, tr, te = be.outer_splits(groups, n_repeats=1)[0]
Xtr, Xte = m.iloc[tr][bf.ALL_FEATURES], m.iloc[te][bf.ALL_FEATURES]
for s in ['A', 'B', 'C']:
    for kind in ['linear', 'tree']:
        pre = bf.preprocessor(s, kind).fit(Xtr)
        print(f'set {s}, {kind:6s}: {len(pre.get_feature_names_out()):2d} model inputs')
pre = bf.preprocessor('A', 'linear').fit(Xtr)
enc = pre.named_transformers_['cat']
print('\nneighbourhood categories learned on this training fold:', list(enc.categories_[0]))
print('grouped as infrequent (< 2 training properties):', list(enc.infrequent_categories_[0]))
unseen = sorted(set(Xte['neighborhood_group']) - set(Xtr['neighborhood_group']))
print('neighbourhoods in the test fold never seen in training:', unseen, '-> encoded as "infrequent", no error')

set A, linear: 17 model inputs
set A, tree  : 17 model inputs
set B, linear: 23 model inputs
set B, tree  : 23 model inputs
set C, linear: 35 model inputs
set C, tree  : 35 model inputs



neighbourhood categories learned on this training fold: ['Adok', 'Babilonija', 'Babin Do', 'Bečići', 'Budva', 'Bulevar / Nivel', 'Centar', 'Dubovica', 'Elmosove zgrade', 'Golubovina', 'Gospoština', 'Mainski put', 'Obilaznica', 'Old Bakery', 'Petrovac', 'Podkošljun', 'Rafailovići', 'Rozino', 'Sveti Stefan', 'Zavala']
grouped as infrequent (< 2 training properties): ['Adok', 'Babin Do', 'Budva', 'Centar', 'Golubovina', 'Gospoština']
neighbourhoods in the test fold never seen in training: ['Podmaine', 'Prijevor - Seoce'] -> encoded as "infrequent", no error


**What this code does:** Shows that a scaler fitted on the training fold differs from one fitted on all rows.

**Why we need it:** It shows concretely what "fit on the training fold only" changes: the test rows never influence the transformation.

In [6]:
pre_train = bf.preprocessor('C', 'linear').fit(Xtr)
pre_all = bf.preprocessor('C', 'linear').fit(m[bf.ALL_FEATURES])
mean_train = pre_train.named_transformers_['num'].mean_
mean_all = pre_all.named_transformers_['num'].mean_
print(pd.DataFrame({'training-fold mean': mean_train, 'all-rows mean (never used)': mean_all},
                   index=pre_train.named_transformers_['num'].feature_names_in_).round(3).to_string())

                        training-fold mean  all-rows mean (never used)
appraisal_year                    2024.375                    2024.212
bedrooms                             1.719                       1.675
bathrooms                            1.312                       1.275
floor                                2.328                       2.288
building_age_years                  12.172                      12.425
elevator                             0.750                       0.738
terrace                              0.969                       0.962
furnished                            0.031                       0.025
renovated                            0.062                       0.050
building_permit                      0.984                       0.975
registered_encumbrance               0.125                       0.112


## 4. Baselines are fitted inside the fold as well

The neighbourhood baseline (B2) uses a neighbourhood's own median €/m² only if that neighbourhood has **at least 3 training properties**. Otherwise it falls back to the Budva median €/m² (B1).

**What this code does:** Fits B2 on the training fold and lists which neighbourhoods get their own rate.

**Why we need it:** It shows how often the baseline really uses neighbourhood information and that it learns only from training rows.

In [7]:
b2 = PricePerM2Baseline('neighbourhood', min_n=3).fit(Xtr, y[tr])
print('Budva median €/m² (training fold): €{:,.0f}'.format(b2.rate_))
print('neighbourhoods with their own rate:', {k: round(v) for k, v in b2.nb_rate_.items()})
share = Xte['neighborhood_group'].isin(b2.nb_rate_).mean()
print(f'test properties priced with their own neighbourhood rate: {100 * share:.0f}%; the rest use the Budva rate')
b3 = PricePerM2Baseline('year', min_year_n=5).fit(Xtr, y[tr])
print('year rates (training fold):', {k: round(v) for k, v in b3.year_rate_.items()})

Budva median €/m² (training fold): €2,314
neighbourhoods with their own rate: {'Babilonija': 1719, 'Bečići': 2124, 'Bulevar / Nivel': 2970, 'Old Bakery': 2721, 'Podkošljun': 1804, 'Rafailovići': 3974, 'Rozino': 2200}
test properties priced with their own neighbourhood rate: 31%; the rest use the Budva rate
year rates (training fold): {2022: 1623, 2023: 2123, 2025: 2382, 2026: 2822}


## 5. The grouped cross-validation folds

- **Outer folds:** `GroupKFold(5, shuffle=True)` on the 73 valuation clusters, repeated 10 times with seeds 0-9.
- **Inner folds:** the same scheme applied to each outer training part; used only for tuning.
- **Every baseline and model is evaluated on these same folds.**

**What this code does:** Builds the 50 outer folds, checks that no cluster or report is split, and saves the assignment.

**Why we need it:** Grouping is the core protection against leakage between related appraisals.

In [8]:
splits = be.outer_splits(groups)
rows = []
for r, k, tr_, te_ in splits:
    assert not set(groups[tr_]) & set(groups[te_]), 'cluster split between training and test'
    assert not set(m['report_group'].iloc[tr_]) & set(m['report_group'].iloc[te_]), 'report split between training and test'
    rows += [{'repeat': r, 'fold': k, 'property_id': m['property_id'].iloc[i]} for i in te_]
folds = pd.DataFrame(rows).sort_values(['repeat', 'property_id'])
assert folds.groupby('repeat')['property_id'].nunique().eq(len(m)).all()
folds.to_csv(bd.RESULTS_DIR / 'cv_folds.csv', index=False, encoding='utf-8', lineterminator='\n')
sizes = pd.Series([len(te_) for *_, te_ in splits])
print(f'{len(splits)} outer folds; test fold size {sizes.min()}-{sizes.max()} properties; every property is tested once per repeat')
print('no valuation cluster and no report is ever split between training and test')

50 outer folds; test fold size 14-21 properties; every property is tested once per repeat
no valuation cluster and no report is ever split between training and test


## 6. Summary

- **Feature sets:** A, B and C are fixed. `air_conditioning`, `orientation`, `construction_year` and `city` are excluded for documented reasons.
- **Forbidden columns:** the target, `price_per_m2_eur`, the identifiers, the grouping columns and `value_basis` are never features.
- **Leakage protection:** all encoders, scalers and baseline statistics are learned inside each training fold.
- **Folds:** the folds are grouped by valuation cluster and saved.
- **Expectation, written before modelling:** most of the information in sets B and C overlaps with area, neighbourhood and year, so it is uncertain whether they help.